# TandemSpec on a Colab T4

**Companion draft adapters for multi-tenant speculative decoding.**

A shared speculative drafter imitates the *base* model, but every request in a multi-LoRA server is
verified against *base + tenant adapter*. This notebook measures the acceptance loss that causes, then
repairs it with a rank-4 LoRA on the drafter trained by on-policy distillation from the adapted target.

Sized for a 16 GB T4 (sm_75): **fp16 and SDPA, not bf16 or FlashAttention-2.**
Default pair is Qwen2.5-1.5B-Instruct (target) + Qwen2.5-0.5B-Instruct (drafter), ~4.2 GB resident,
leaving room for QLoRA tenant training on the same card.

Runtime -> Change runtime type -> **T4 GPU** before running.


## 0 - Environment


In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name}  sm_{p.major}{p.minor}  {p.total_memory/1e9:.1f} GB')
    if p.major < 8:
        print('sm < 80: use fp16 + SDPA. bf16 and FlashAttention-2 are unavailable here.')


In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes datasets


## 1 - Get the repo

Upload the `tandemspec/` folder to the Colab file browser, or unzip it here.


In [ ]:
import os, sys
REPO = '/content/tandemspec'   # adjust if you unzipped elsewhere
assert os.path.isdir(REPO), 'upload or unzip the tandemspec repo to /content/tandemspec'
sys.path.insert(0, REPO)
os.chdir(REPO)
print(os.listdir('.'))


## 2 - Verify the measurement machinery first

Before trusting any acceptance number, check that the simulator (a) reproduces the target distribution
exactly -- speculative decoding is lossless -- and (b) matches the closed-form acceptance rate
`beta = sum_v min(p_v, q_v)`. Everything downstream rests on this.


In [ ]:
!python tests/test_accept.py


## 3 - Mint tenant adapters with QLoRA

One rank-16 adapter per task domain, trained against an **nf4-quantised** base -- the ordinary way a
tenant fine-tunes. ~10 min for 4 tenants at 300 steps on a T4.


In [ ]:
!python experiments/gpu/tandemspec_gpu.py --stage tenants \
    --n_tenants 4 --tenant_steps 300 --task_rank 16 --out /content/ts_out


## 4 - E1: the acceptance collapse

Sweeps a runtime strength multiplier on each tenant adapter from 0 (base model) to 1, regenerating
evaluation rollouts from the adapted target at every point so the drafter is always measured on the
distribution it would actually see.


In [ ]:
!python experiments/gpu/tandemspec_gpu.py --stage e1 \
    --n_tenants 4 --n_eval 24 --gen_len 96 --strengths 0,0.25,0.5,0.75,1.0 --out /content/ts_out


## 5 - E2: companion draft adapters

Trains a rank-4 LoRA on the **drafter** for each tenant, distilling from the adapted target on rollouts
sampled from it. Two objectives are compared: `tvd` (which is exactly `1 - beta`, so minimising it
maximises acceptance) and `fkl` (conventional distillation).


In [ ]:
!python experiments/gpu/tandemspec_gpu.py --stage e2 \
    --n_tenants 4 --companion_rank 4 --companion_steps 200 --losses tvd,fkl \
    --n_eval 24 --n_roll 96 --out /content/ts_out


## 6 - Measured cost ratio

The step-cost model needs one empirical number: how much a drafter forward costs relative to a target
forward at decode. Everything else follows from measured acceptance.


In [ ]:
!python experiments/gpu/tandemspec_gpu.py --stage throughput --out /content/ts_out


## 7 - Results


In [ ]:
import json, statistics as st
e1 = json.load(open('/content/ts_out/gpu_e1.json'))
e2 = json.load(open('/content/ts_out/gpu_e2.json'))
c  = json.load(open('/content/ts_out/gpu_cost_ratio.json'))

print('E1 - acceptance vs adapter strength')
for t in sorted({r['tenant'] for r in e1}):
    rows = sorted([r for r in e1 if r['tenant']==t], key=lambda r: r['strength'])
    b0, b1 = rows[0]['beta'], rows[-1]['beta']
    print(f"  {t:<8} beta {b0:.4f} -> {b1:.4f}  ({100*(b1-b0)/b0:+.1f}%)   "
          f"greedy {rows[0]['beta_greedy']:.4f} -> {rows[-1]['beta_greedy']:.4f}")

print('\nE2 - repair')
for arm in sorted({r['arm'] for r in e2}):
    v = [r for r in e2 if r['arm']==arm]
    print(f"  {arm:<24} beta {st.mean(r['beta'] for r in v):.4f}   "
          f"tokens/step {st.mean(r['tokens_per_step_iid'] for r in v):.3f}   "
          f"params {v[0]['params']:,}")

print('\ncost ratio (draft forward / target forward):', round(c['cost_ratio_measured'], 4))


## 8 - Push measured acceptance through the serving model


In [ ]:
from tandemspec.eval.throughput import SCENARIOS, speedup, best_gamma
import json, statistics as st
e2 = json.load(open('/content/ts_out/gpu_e2.json'))
shared = st.mean(r['beta'] for r in e2 if r['arm']=='shared-drafter')
comp   = st.mean(r['beta'] for r in e2 if r['arm'].startswith('companion-tvd'))
print(f'beta: shared {shared:.4f} -> companion {comp:.4f}\n')
for s in SCENARIOS:
    a, b = speedup(shared, 4, s.cost_ratio, s.mode), speedup(comp, 4, s.cost_ratio, s.mode)
    print(f'{s.name:<42} {a:.2f}x -> {b:.2f}x   optimal gamma '
          f'{best_gamma(shared, s.cost_ratio, s.mode)[0]} -> {best_gamma(comp, s.cost_ratio, s.mode)[0]}')


## 9 - What vLLM can and cannot do today

This is the gap the project targets: the proposer has no LoRA plumbing, so whatever adapter a request
carries is applied to the target and ignored by the drafter.


In [ ]:
from tandemspec.serving.vllm_integration import check_vllm_support
print(check_vllm_support().summary())


### Bigger pair, if you have more memory

```bash
python experiments/gpu/tandemspec_gpu.py --stage e1 \
    --target Qwen/Qwen2.5-7B-Instruct --draft Qwen/Qwen2.5-0.5B-Instruct \
    --four_bit_serve --n_eval 16 --eval_bs 1
```

`--four_bit_serve` also serves the target 4-bit, which makes the QLoRA train/serve quantisation
*match*; dropping it reproduces the mismatch arm of E4.
